In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,END,START
from langgraph.types import interrupt, Command
from rich import print


#1. 状態を宣言
class OverAllState(TypedDict):
    initial_state:str
    parallel_node_a_1:str
    parallel_node_a_2:str
    node_b_output:str

#2. ノードを宣言
def parallel_node_a_1(state:OverAllState) -> OverAllState:
    return {
        "parallel_node_a_1":"並行ノードA-1の出力"
    }

def parallel_node_a_2(state:OverAllState) -> OverAllState:
    return {
        "parallel_node_a_2":"並行ノードA-2の出力"
    }

def node_b(state:OverAllState) -> OverAllState:
    # interrupt("hello")
    return {
        "node_b_output":"ノードBの出力"
    }

#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("parallel_node_a_1",parallel_node_a_1)
builder.add_node("parallel_node_a_2",parallel_node_a_2)
builder.add_node("node_b",node_b)
builder.add_edge(START,"parallel_node_a_1")
builder.add_edge(START,"parallel_node_a_2")
builder.add_edge(["parallel_node_a_1","parallel_node_a_2"],"node_b")
builder.add_edge("node_b",END)

#4. チェックポインターバックエンドを追加
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer= checkpointer)

from IPython.display import display
display(graph)

config = {"configurable":{"thread_id":"123"}}
for chunk in graph.stream(
    {"initial_state":"初期状態"},
    config=config,
    stream_mode=["debug"],
    version="v2"
    # stream_mode=["tasks"]
    # stream_mode=["checkpoints"]
):
    print(chunk)


In [ ]:
from langgraph.types import interrupt, Command
print("*" * 50 )
for chunk in graph.stream(
    Command(resume=""),
    config=config,
    stream_mode=["checkpoints"]
):
    print(chunk)


In [ ]:
print(graph.get_state_history(config=config))